# qlib Native Research Notebook

这份 notebook 现在只负责两件事：
- 配置并调用统一的 `qlib_native` 研究 runner。
- 读取统一产物，展示研究价值、执行质量、风险和稳定性诊断。

训练、打分、原生回测、shared-signal validation、diagnostics 落盘，都由脚本化模块负责，不再在 notebook 内重复维护一条平行逻辑。

## 1. 目标与使用方式

建议的使用姿势：
- 日常研究时，把 `RUN_WORKFLOW = False`，直接复盘已有产物。
- 需要刷新 baseline / candidate 时，把 `RUN_WORKFLOW = True`，让 notebook 直接调用统一 runner。
- 所有主流程参数都集中在一个 `CONFIG` 字典里，避免 notebook 和脚本参数漂移。

## 2. 环境检查与工作流入口

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from qlib_research.core.notebook_workflow import (
    load_native_workflow_artifacts,
    run_native_notebook_workflow,
)


def _safe_last_numeric(frame: pd.DataFrame, column: str):
    if frame.empty or column not in frame.columns:
        return None
    series = pd.to_numeric(frame[column], errors="coerce").dropna()
    if series.empty:
        return None
    return float(series.iloc[-1])


def _safe_max_numeric(frame: pd.DataFrame, column: str):
    if frame.empty or column not in frame.columns:
        return None
    series = pd.to_numeric(frame[column], errors="coerce").dropna()
    if series.empty:
        return None
    return float(series.max())


def _show_line(frame: pd.DataFrame, x: str, y, title: str, height: int = 360):
    if frame.empty:
        display(Markdown(f"_没有可展示的数据：{title}_"))
        return
    fig = px.line(frame, x=x, y=y, title=title)
    fig.update_layout(template="plotly_white", height=height)
    fig.show()


def _prepare_heatmap_frame(heatmap_frame: pd.DataFrame) -> pd.DataFrame:
    if heatmap_frame.empty:
        return heatmap_frame
    frame = heatmap_frame.copy()
    if "year" in frame.columns:
        frame["year"] = pd.to_numeric(frame["year"], errors="coerce")
        frame = frame.loc[frame["year"].notna()].copy()
        frame["year"] = frame["year"].astype(int).astype(str)
        frame = frame.set_index("year")
    if "Unnamed: 0" in frame.columns:
        frame = frame.set_index("Unnamed: 0")
    frame = frame.apply(pd.to_numeric, errors="coerce")
    return frame


def _render_return_heatmap(heatmap_frame: pd.DataFrame, title: str, x_label: str, y_label: str) -> None:
    normalized_frame = _prepare_heatmap_frame(heatmap_frame)
    if normalized_frame.empty:
        display(Markdown(f"_没有可展示的数据：{title}_"))
        return
    numeric_frame = normalized_frame * 100.0
    finite_values = numeric_frame.to_numpy().ravel()
    finite_values = finite_values[pd.notna(finite_values)]
    if len(finite_values):
        abs_values = pd.Series(finite_values).abs()
        robust_bound = float(abs_values.quantile(0.90)) if not abs_values.empty else 1.0
        robust_bound = max(robust_bound, 1.0)
        symmetric_bound = min(robust_bound, 30.0)
    else:
        symmetric_bound = 1.0
    fig = px.imshow(
        numeric_frame,
        text_auto=".1f",
        aspect="auto",
        color_continuous_scale=[
            [0.0, "#b2182b"],
            [0.5, "#f7f7f7"],
            [1.0, "#1a9850"],
        ],
        zmin=-symmetric_bound,
        zmax=symmetric_bound,
        origin="lower",
        title=title,
        labels={"x": x_label, "y": y_label, "color": "Net Return %"},
    )
    fig.update_layout(height=420)
    fig.show()


runtime_check = pd.DataFrame(
    [
        {"check": "PROJECT_ROOT", "value": str(PROJECT_ROOT)},
        {"check": "Notebook role", "value": "thin native workflow viewer"},
        {"check": "Runner helper", "value": "run_native_notebook_workflow"},
        {"check": "Artifact loader", "value": "load_native_workflow_artifacts"},
    ]
)
display(runtime_check)


## 3. 参数区

In [ ]:
RUN_WORKFLOW = False
RECIPE_NAMES = ["baseline", "mae_4w", "binary_4w", "rank_blended", "huber_8w"]
FOCUS_RECIPE = "rank_blended"

CONFIG = {
    "universe_profile": "csi300",
    "panel_path": "artifacts/panels/csi300_weekly_20260410.parquet",
    "execution_panel_path": None,
    "output_dir": "artifacts/native_workflow/csi300_4w_allfeatures_20260410",
    "run_export": "auto_if_missing",
    "start_date": "2016-01-01",
    "end_date": None,
    "batch_size": 200,
    "topk": 10,
    "train_weeks": 260,
    "valid_weeks": 52,
    "eval_count": 52,
    "step_weeks": 4,
    "walk_forward_enabled": True,
    "walk_forward_eval_count": 0,
    "walk_forward_train_weeks": 260,
    "walk_forward_valid_weeks": 52,
    "walk_forward_step_weeks": 4,
    "benchmark_mode": "auto",
    "signal_objective": "huber_regression",
    "label_recipe": "blended_excess_4w_8w",
    "rebalance_interval_weeks": 4,
    "validation-execution-lag-steps": 1,
    "hold_buffer_rank": 15,
    "universe_exit_policy": "retain_quotes_for_existing_positions",
    "min_liquidity_filter": 0.0,
    "min_score_spread": 0.0,
    "industry_max_weight": 0.30,
    "diagnostics_enabled": True,
    "run_validation_comparison": True,
    "publish_model": False,
}

display(pd.DataFrame([CONFIG]).T.rename(columns={0: "value"}))
display(Markdown("`benchmark_mode` 默认按 `universe_profile` 自动映射；也可以手工设为 `000001.SH` 或 `000001.SH@上证指数`。"))
display(Markdown("`walk_forward_eval_count=0` 表示使用全部可评估周；大于 0 时只保留最近 N 个评估周。"))


## 4. 运行或加载统一研究工作流

In [ ]:
workflow_run = run_native_notebook_workflow(
    config_overrides=CONFIG,
    recipe_names=RECIPE_NAMES,
    run_workflow=RUN_WORKFLOW,
)

artifact_view = load_native_workflow_artifacts(
    workflow_run["output_dir"],
    recipe_names=RECIPE_NAMES,
)
workflow_summary = workflow_run["workflow_summary"]

display(Markdown(f"**输出目录**: `{workflow_run['output_dir']}`"))
display(pd.DataFrame([workflow_summary.get("config", CONFIG)]).T.rename(columns={0: "value"}))

if artifact_view["promotion_gate"].empty:
    display(Markdown("当前没有可展示的 promotion gate 输出；如果只运行了 `baseline`，或还没有刷新 workflow，这是正常的。"))
else:
    display(artifact_view["promotion_gate"])


## 5. 工作流总览

In [ ]:
recipe_overview = artifact_view["recipe_overview"].copy()
if recipe_overview.empty:
    display(Markdown("当前输出目录下还没有 recipe 产物。把 `RUN_WORKFLOW` 改成 `True` 后重新运行上一格。"))
else:
    recipe_overview = recipe_overview.sort_values("recipe").reset_index(drop=True)
    walk_forward_limit = workflow_summary.get("config", {}).get("walk_forward_eval_count")
    if pd.notna(walk_forward_limit) and int(walk_forward_limit) > 0:
        display(Markdown(f"当前 `walk_forward_eval_count={int(walk_forward_limit)}`，只会保留最近 {int(walk_forward_limit)} 个可评估周，而不是全历史 walk-forward。"))
    key_metric_columns = [
        "recipe",
        "rolling_rank_ic_ir",
        "rolling_topk_mean_excess_return_4w",
        "rolling_eval_date_count",
        "rolling_eval_date_start",
        "rolling_eval_date_end",
        "rolling_net_total_return",
        "rolling_max_drawdown",
        "walk_forward_rank_ic_ir",
        "walk_forward_topk_mean_excess_return_4w",
        "walk_forward_eval_date_count",
        "walk_forward_eval_date_start",
        "walk_forward_eval_date_end",
        "walk_forward_net_total_return",
        "walk_forward_max_drawdown",
        "latest_score_dispersion",
        "latest_top10_unique_score_count",
        "latest_actual_hold_count",
        "latest_blocked_sell_count",
    ]
    available_columns = [column for column in key_metric_columns if column in recipe_overview.columns]
    key_metric_frame = recipe_overview[available_columns].copy()
    display(key_metric_frame)
    display(recipe_overview)
    display(Markdown("上表是集中指标看板；`recipe_overview` 则保留完整细项。`requested_feature_count` 对应配置请求特征数，`used_feature_count` 对应实际进入训练的特征数。"))


## 6. 研究价值看板

In [ ]:
available_recipes = artifact_view["recipe_names"]
focus_recipe = FOCUS_RECIPE if FOCUS_RECIPE in available_recipes else (available_recipes[0] if available_recipes else None)

if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    feature_prefilter = recipe_data["feature_prefilter"]
    signal_diagnostics = recipe_data["signal_diagnostics"].copy()
    portfolio_diagnostics = recipe_data["portfolio_diagnostics"].copy()
    slice_regime_summary = recipe_data["slice_regime_summary"].copy()

    research_value_board = pd.DataFrame(
        [
            {
                "recipe": focus_recipe,
                "requested_feature_count": _safe_max_numeric(feature_prefilter, "requested_feature_count"),
                "used_feature_count": _safe_max_numeric(feature_prefilter, "selected_feature_count"),
                "latest_score_dispersion": _safe_last_numeric(signal_diagnostics, "score_dispersion"),
                "latest_top10_unique_score_count": _safe_last_numeric(signal_diagnostics, "topk_unique_score_count"),
                "latest_top10_unique_score_ratio": _safe_last_numeric(signal_diagnostics, "topk_unique_score_ratio"),
                "latest_actual_hold_count": _safe_last_numeric(portfolio_diagnostics, "actual_hold_count"),
                "latest_blocked_sell_count": _safe_last_numeric(portfolio_diagnostics, "blocked_sell_count"),
                "slice_row_count": int(len(slice_regime_summary)),
            }
        ]
    )
    display(research_value_board)

    if not signal_diagnostics.empty and "feature_date" in signal_diagnostics.columns:
        signal_diagnostics["feature_date"] = pd.to_datetime(signal_diagnostics["feature_date"])
        _show_line(
            signal_diagnostics,
            x="feature_date",
            y=["score_dispersion", "topk_unique_score_ratio"],
            title=f"{focus_recipe}: 分数离散度与 TopK 唯一分数比例",
        )


## 7. 执行与风险诊断

In [ ]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    portfolio_diagnostics = recipe_data["portfolio_diagnostics"].copy()
    execution_diff_summary = recipe_data["execution_diff_summary"].copy()
    rolling_native_report = recipe_data["rolling_native_report"].copy()
    walk_forward_native_report = recipe_data["walk_forward_native_report"].copy()
    rolling_monthly_heatmap = recipe_data["rolling_native_monthly_return_heatmap"].copy()
    rolling_annual_heatmap = recipe_data["rolling_native_annual_return_heatmap"].copy()
    walk_forward_monthly_heatmap = recipe_data["walk_forward_native_monthly_return_heatmap"].copy()
    walk_forward_annual_heatmap = recipe_data["walk_forward_native_annual_return_heatmap"].copy()

    display(portfolio_diagnostics.tail(12) if not portfolio_diagnostics.empty else pd.DataFrame())
    display(execution_diff_summary if not execution_diff_summary.empty else pd.DataFrame())

    if not portfolio_diagnostics.empty and "trade_date" in portfolio_diagnostics.columns:
        portfolio_diagnostics["trade_date"] = pd.to_datetime(portfolio_diagnostics["trade_date"])
        _show_line(
            portfolio_diagnostics,
            x="trade_date",
            y=["target_hold_count", "actual_hold_count", "blocked_sell_count"],
            title=f"{focus_recipe}: 目标持仓、实际持仓与 blocked sell",
        )

    if not rolling_native_report.empty and "datetime" in rolling_native_report.columns:
        rolling_native_report["datetime"] = pd.to_datetime(rolling_native_report["datetime"])
        _show_line(
            rolling_native_report,
            x="datetime",
            y=["net_value", "benchmark_value"],
            title=f"{focus_recipe}: rolling 净值 vs benchmark",
            height=420,
        )

    if not walk_forward_native_report.empty and "datetime" in walk_forward_native_report.columns:
        walk_forward_native_report["datetime"] = pd.to_datetime(walk_forward_native_report["datetime"])
        _show_line(
            walk_forward_native_report,
            x="datetime",
            y=["net_value", "benchmark_value"],
            title=f"{focus_recipe}: walk-forward 净值 vs benchmark",
            height=420,
        )

    _render_return_heatmap(
        rolling_monthly_heatmap,
        title=f"{focus_recipe}: rolling 月度净收益热力图",
        x_label="Month",
        y_label="Year",
    )
    _render_return_heatmap(
        rolling_annual_heatmap,
        title=f"{focus_recipe}: rolling 年度净收益热力图",
        x_label="Year",
        y_label="Metric",
    )
    _render_return_heatmap(
        walk_forward_monthly_heatmap,
        title=f"{focus_recipe}: walk-forward 月度净收益热力图",
        x_label="Month",
        y_label="Year",
    )
    _render_return_heatmap(
        walk_forward_annual_heatmap,
        title=f"{focus_recipe}: walk-forward 年度净收益热力图",
        x_label="Year",
        y_label="Metric",
    )


## 8. 切片稳定性与特征重要性

In [ ]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    recipe_data = artifact_view["recipes"][focus_recipe]
    slice_regime_summary = recipe_data["slice_regime_summary"].copy()
    rolling_feature_importance = recipe_data["rolling_feature_importance"].copy()

    if not slice_regime_summary.empty:
        display(slice_regime_summary.sort_values(["bundle", "slice_type", "coverage"], ascending=[True, True, False]).head(30))
    else:
        display(pd.DataFrame())

    if not rolling_feature_importance.empty:
        top_features = (
            rolling_feature_importance.groupby("feature", as_index=False)["importance_gain"]
            .mean()
            .sort_values("importance_gain", ascending=False)
            .head(20)
        )
        display(top_features)
        fig = px.bar(
            top_features.sort_values("importance_gain", ascending=True),
            x="importance_gain",
            y="feature",
            orientation="h",
            title=f"{focus_recipe}: rolling 平均特征重要性（gain）",
        )
        fig.update_layout(template="plotly_white", height=520)
        fig.show()


## 9. 最新一期信号快照

In [ ]:
if focus_recipe is None:
    display(Markdown("当前没有可展示的 recipe。"))
else:
    latest_score_frame = artifact_view["recipes"][focus_recipe]["latest_score_frame"].copy()
    if latest_score_frame.empty:
        display(Markdown("当前 recipe 没有最新一期快照。"))
    else:
        display(latest_score_frame.head(20))
        if {"l1_name", "score"}.issubset(latest_score_frame.columns):
            industry_snapshot = (
                latest_score_frame.head(50)
                .groupby("l1_name", dropna=False)
                .agg(stock_count=("instrument", "count"), mean_score=("score", "mean"))
                .reset_index()
                .sort_values(["mean_score", "stock_count"], ascending=[False, False])
            )
            display(industry_snapshot)

display(Markdown("如果要发布线上 snapshot，请把 `CONFIG['publish_model']` 改成 `True`，然后重新运行 workflow cell。"))


## 10. 后续动作

建议优先按这个顺序继续推进：
- 先比较 `baseline` 与 candidate 的 `execution_diff_summary`，确认收益问题到底来自信号还是执行。
- 再看 `research_value_board` 与 `slice_regime_summary`，确认分数粒度、年份稳定性、行业集中度是否改善。
- 最后才决定是否发布 snapshot，避免把“会排不会赚”的配方直接推向下游。